In [19]:
import sys

sys.path.append("twin")

In [20]:
import json
import os
from pathlib import Path

import gradio as gr
import requests
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from rich.console import Console
from rich.markdown import Markdown
from twin.style import CUSTOM_CSS as css

# Keep this aligned with the actual project root used by the notebook
SCRIPT_DIR = Path.cwd().resolve()


In [21]:
console = Console()

In [22]:
def get_env_file(start: Path) -> Path | None:
    """Search for a .env file in the current directory and parent directories."""
    for parent in [start, *start.parents]:
        candidate = parent / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env file found in the current directory or parent directories.")


load_dotenv(dotenv_path=get_env_file(SCRIPT_DIR), override=True)

True

In [23]:
openai = OpenAI()


## Setup Pushover


In [27]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Correct Pushover user token")
    else:
        print("Invalid token")
else:
    print("Pushover token not available")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Correct Pushover user token")
    else:
        print("Invalid token")
else:
    print("Pushover token not available")

Correct Pushover user token
Correct Pushover user token


In [28]:
def push(message: str) -> None:
    """Send a notification message through the configured Pushover account."""
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

    return None

In [29]:
push("Hi there")

Push: Hi there


In [30]:
def record_user_details(
    email: str, name: str = "Name not provided", notes: str = "Not provided"
) -> str:
    """Record a user's contact details and interest notes for follow-up."""
    push(f"Recording interests from {name} with email {email} and notes {notes}")
    return "OK"

In [31]:
def record_unknown_question(question: str) -> str:
    """Record a user question that the agent could not answer."""
    push(f"Recording question {question} asked that I don't know")
    return "OK"

In [32]:
record_user_details_json = {
    "name": "record_user_details",
    "description": (
        "Use this tool to record that a user is interested in being in "
        "touch and provided an email address"
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user",
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it",
            },
            "notes": {
                "type": "string",
                "description": (
                    "Any additional info about the conversation that's worth "
                    "recording to give context"
                ),
            },
        },
        "required": ["email"],
        "additionalProperties": False,
    },
}

In [33]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": (
        "Always use this tool to record any question that couldn't be "
        "answered as you didn't know the answer"
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

In [34]:
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json},
]

In [35]:
# Function to handle multiple tool calls by the agent
def handle_tool_calls(tool_calls: list[object]) -> list[dict]:
    """Execute agent-requested tool calls and format the tool responses.

    Args:
        tool_calls: A list of OpenAI-style tool call objects with a function
            name and JSON-encoded arguments.

    Returns:
        A list of tool-role response dictionaries ready to send back to the
        model, including the tool call ID and serialized result payload.

    Raises:
        ValueError: If any tool call arguments are not valid JSON.
    """
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        if tool:
            result = tool(**arguments)
        else:
            console.print(f"[red]Tool {tool_name} not found[/red]")
            result = {"error": f"Tool {tool_name} not found"}
        results.append(
            {"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id}
        )
    return results

In [36]:
reader = PdfReader(SCRIPT_DIR / "twin" / "Profile.pdf")
linkedin = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open(SCRIPT_DIR / "twin" / "summary.txt", encoding="utf-8") as f:
    summary = f.read()

In [37]:
# Set up the initial prompt to set the context. Role is "developer" or "system"

system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.
"""  # noqa: E501


In [38]:
console.print(Markdown(system_prompt))

Your role                                                     

You are a digital twin running on a website, chatting with visitors of the website. You represent the person who's 
website you are on. You answer questions related to their career, background, skills and experience.               

Here are the details of the person you are representing:                                                           

I'm Anirban Guha, interested in movies, stories, theaters. I am in corporate world. Interested in learning new     
things, and in the present days, AI.                                                                               

If asked, you explain clearly that you are an AI that is the digital twin of this person.                          

                                                      Context                                                      

Here is a summary of the person's LinkedIn profile so that you can answer questions:                               

Contact anirban.guha@tcs.com www.linkedin.com/in/anirbanguha (LinkedIn) www.tcs.com (Company) Top Skills Operations
Management Analytical Skills Coaching Certifications Predicting Bank Customer Churn Data Science in Python:        
Unsupervised Learning Natural Language Processing in Python Python Basics Python Functions, Files, and Dictionaries
Anirban Guha Retail Cluster Delivery Head at Tata Consultancy Services Kolkata, West Bengal, India Summary I am    
working in Tata Consultancy Services Ltd., the leading software consulting company in India for the last 11 years. 
Having a good experience in Program and Relationship Management for a number of industires. Specialties:           
Publishing, Media, Broadcasting, Entertainment Experience Tata Consultancy Services 24 years Head Of Service       
Delivery 2014 - Present (12 years) Business Relationship Manager June 2008 - 2014 (6 years) Client Partner 2008 -  
2014 (6 years) Program Manager October 2002 - June 2008 (5 years 9 months) Project Manager 2002 - 2004 (2 years)   
Education Jadavpur University BCSE, Computer Science · (1993 - 1997) South Point High School   Page 1 of 1         

                                                       Rules                                                       

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came
across the website. Only answer questions related to career, background, skills and experience. If the user asks   
about something unrelated, then steer the conversation back to professional topics.                                

Always stay in character as the digital twin of the person you are representing. Represent the person.             

If the user would like to get in touch, then ask for their email, and use your tool to record their email for      
follow-up.                                                                                                         

IMPORTANT: If you don't know the answer, use your tool to record the question, and then tell the user that you     
don't know. Never make up an answer.

In [39]:
# Agent Loop


def chat(message: str, history: list[dict]) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        *history,
        {"role": "user", "content": message},
    ]

    response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(
            model="gpt-5.4-nano", messages=messages, tools=tools
        )

    return response.choices[0].message.content

In [40]:
gr.ChatInterface(chat, title="Digital Twin").launch(inbrowser=True, css=css)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
